# 03 - Optimize with DSPy

This notebook runs DSPy optimizers to improve translation quality:
1. **BootstrapFewShot** — selects best few-shot examples (start here)
2. **MIPROv2** — jointly optimizes instructions + examples (for larger datasets)
3. **BootstrapFinetune** — fine-tunes a local model (requires GPU)

**Prerequisites:** Run `01_parse_data.ipynb` and `02_prototype.ipynb` first.

In [8]:
import sys
import os
import importlib
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Reload all src modules to pick up code changes
import src.retriever, src.signatures, src.modules, src.metrics, src.optimize
for mod in [src.retriever, src.signatures, src.modules, src.metrics, src.optimize]:
    importlib.reload(mod)

import dspy
from src.optimize import (
    setup_lm,
    optimize_bootstrap_fewshot,
    optimize_mipro,
    evaluate_program,
    save_program,
    load_program,
)
from src.modules import TranslationChatbot, format_response
from src.metrics import load_examples, translation_quality_metric

print(f'Project root: {project_root}')

Project root: c:\Users\gioan\Documents\GitHub\chatbutt


## Step 1: Setup

In [9]:
# Configure LM
api_key = os.getenv('GROQ_API_KEY')
if api_key and api_key != 'gsk_your-key-here':
    lm = setup_lm(provider='groq', model='llama-3.3-70b-versatile', api_key=api_key)
else:
    lm = setup_lm(provider='ollama', model='llama3.2')

# Load train/dev sets
trainset = load_examples(project_root / 'data' / 'examples' / 'trainset.json')
devset = load_examples(project_root / 'data' / 'examples' / 'devset.json')

print(f'Trainset: {len(trainset)} examples')
print(f'Devset: {len(devset)} examples')

# Create base chatbot
chroma_dir = str(project_root / 'chroma_db')
chatbot = TranslationChatbot(chroma_dir=chroma_dir)

Configured DSPy with groq/llama-3.3-70b-versatile
Trainset: 50 examples
Devset: 25 examples


## Step 2: Evaluate Base (Unoptimized) Performance

In [ ]:
print('Evaluating base (unoptimized) chatbot...')
base_score = evaluate_program(chatbot, devset)
print(f'Base score: {base_score:.2f}%')

Evaluating base (unoptimized) chatbot...
Average Metric: 19.00 / 25 (76.0%): 100%|██████████| 25/25 [00:08<00:00,  2.93it/s]

2026/02/11 12:45:56 INFO dspy.evaluate.evaluate: Average Metric: 19.0 / 25 (76.0%)



Evaluation score: 76.00%
Base score: 7600.00%


## Step 3: Optimize with BootstrapFewShot

This optimizer automatically selects the best few-shot demonstrations
by bootstrapping from the training set. Good starting point with 10-50 examples.

In [ ]:
print('Running BootstrapFewShot optimization...')
optimized_chatbot = optimize_bootstrap_fewshot(
    chatbot,
    trainset,
    metric=translation_quality_metric,
    max_bootstrapped_demos=4,
    max_labeled_demos=8,
)

print('\nEvaluating optimized chatbot...')
optimized_score = evaluate_program(optimized_chatbot, devset)
print(f'\nBase score:      {base_score:.2f}%')
print(f'Optimized score: {optimized_score:.2f}%')
print(f'Improvement:     {optimized_score - base_score:+.2f}%')

## Step 4: Save the Optimized Program

In [ ]:
save_path = project_root / 'optimized' / 'translation_v1.json'
save_program(optimized_chatbot, save_path)
print(f'Saved to {save_path}')

## Step 5: Test the Optimized Chatbot

In [ ]:
# Load the saved program
loaded = load_program(save_path, chroma_dir=chroma_dir)

test_messages = [
    "Translate 'water'",
    "How do you say 'thank you' in Akeanon?",
    "Translate: I love you",
    "What is 'beautiful' in Hiligaynon?",
    "Translate: Where are you going?",
]

for msg in test_messages:
    print(f'\n{"="*60}')
    print(f'User: {msg}')
    print(f'{"="*60}')
    prediction = loaded(user_query=msg)
    response = format_response(prediction)
    print(response)

## (Optional) Step 6: MIPROv2 Optimization

Use this if you have 200+ examples. It jointly optimizes instructions and demonstrations.
Takes longer but can produce significantly better results.

In [ ]:
# # Uncomment to run MIPROv2 (requires more examples and takes longer)
# print('Running MIPROv2 optimization...')
# mipro_chatbot = optimize_mipro(
#     chatbot,
#     trainset,
#     metric=translation_quality_metric,
#     num_trials=20,
# )
#
# mipro_score = evaluate_program(mipro_chatbot, devset)
# print(f'MIPROv2 score: {mipro_score:.2%}')
#
# save_program(mipro_chatbot, project_root / 'optimized' / 'translation_mipro.json')

## (Optional) Step 7: Fine-Tuning with BootstrapFinetune

This uses a large teacher model (e.g., GPT-4o-mini) to generate training data,
then fine-tunes a small local model. Requires a GPU and more setup.

In [ ]:
# # Uncomment to run BootstrapFinetune (requires GPU + additional packages)
# # pip install torch transformers accelerate trl peft sglang
# dspy.settings.experimental = True
#
# # Configure teacher and student models
# teacher_lm = dspy.LM('openai/gpt-4o-mini')
# student_lm = dspy.LM('openai/local:meta-llama/Llama-3.2-1B-Instruct',
#                       provider=dspy.LocalProvider())
#
# # Run fine-tuning
# with dspy.context(lm=teacher_lm):
#     optimizer = dspy.BootstrapFinetune(
#         metric=translation_quality_metric,
#         target_model=student_lm,
#     )
#     finetuned = optimizer.compile(chatbot, trainset=trainset)
#
# save_program(finetuned, project_root / 'optimized' / 'translation_finetuned.json')

---
**Next:** Run `python app/gradio_app.py` to launch the chatbot UI!